# 03 — Session B fallback training (IndicTrans2 LoRA, Kaggle)

Fallback path when the Session A (Bodhan) model is unavailable. Fine-tunes
`ai4bharat/indictrans2-en-indic-dist-200M` (MIT) with LoRA for
`hin_Deva -> mar_Deva` using the `AI4Bharat/IndicTrans2` huggingface_interface
and the mandatory `IndicTransToolkit` (`IndicProcessor`, `IndicDataCollator`).

Stability guards for the known random-segfault issue (upstream #117):
`transformers>=4.33.2,<5`, `attn_implementation="eager"`, `dataloader_num_workers=0`, fp16.

In [ ]:
# Session B MUST NOT use transformers>=5 (conflicts with IndicTransToolkit).
!pip install -q "transformers>=4.33.2,<5" "indictranstoolkit==1.1.1" peft datasets accelerate sentencepiece sacrebleu huggingface_hub pyyaml

In [ ]:
# Install the IndicTrans2 huggingface_interface (model + custom code) via git.
!git clone --depth 1 https://github.com/AI4Bharat/IndicTrans2 /tmp/IndicTrans2
!pip install -q /tmp/IndicTrans2/huggingface_interface

In [ ]:
# Hugging Face login — token comes from the environment, never hardcoded.
import os

from huggingface_hub import login

token = os.environ.get("HF_TOKEN")
if not token:
    raise RuntimeError("Set the HF_TOKEN environment variable (Kaggle Secrets) before running.")
login(token=token)
print("HF login OK")

In [ ]:
# Run Session B training (Cell B). Writes the adapter to /kaggle/working/cellB/adapter.
!PYTHONPATH=src python -m mr_mt.train_indictrans2_lora --config configs/cellB_indictrans2_lora.yaml

In [ ]:
# Verify the saved adapter.
from pathlib import Path

adapter = Path("/kaggle/working/cellB/adapter")
assert adapter.is_dir(), f"adapter dir missing: {adapter}"
print(sorted(p.name for p in adapter.iterdir()))